# 05 — Exploitability Analysis
Is the above-floor longshot overpricing economically exploitable after Kalshi fees and spreads?

**Strategy:** buy NO on every above-floor longshot (0.02 < snapshot YES price ≤ 0.15),
hold to settlement, equal-weighted across candidates. Event = cluster.
**Data:** `data/candidate_level_discrete.csv` — locked calibration dataset, actual outcomes.

**Reads:** `data/candidate_level_discrete.csv`
**Writes:** console / table output only


In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from utils import (RNG_SEED, BOOT_REPS, FEE_RATE, FLOOR_MAX, LS_MAX,
                   C_GRID, clopper_pearson_upper)

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath('..')
DATA_DIR = os.path.join(BASE_DIR, 'data')

cand = pd.read_csv(os.path.join(DATA_DIR, 'candidate_level_discrete.csv'))
print(f'candidate_level_discrete : {len(cand):,} rows  |  {cand["event"].nunique()} events')
print(f'Columns: {list(cand.columns)}')


candidate_level_discrete : 1,579 rows  |  133 events
Columns: ['ticker', 'event', 'domain', 'won', 'snapshot_price', 'n_trades_before_snapshot', 'staleness_hours', 'at_floor', 'n_event_candidates', 'bucket']


In [2]:
# ── Kalshi fee formula ────────────────────────────────────────────────────
# Source: help.kalshi.com/trading/fees  (Feb 2026 fee schedule)
#
# Taker fee per contract = round_up( 0.07 × C × P × (1 − P) )
# Maker fee per contract = round_up( 0.0175 × C × P × (1 − P) )
#
# P = YES price in dollars; symmetric — same formula whether you buy YES or NO.
# round_up = round to next $0.0001 (centicent) — negligible here; we use unrounded.
# Strategy assumes TAKER (market orders), the conservative case.

print('Kalshi taker fee  =  0.07 × P × (1 − P)  per contract')
print()
print(f'  {"YES price P":>12}  {"NO price (1-P)":>15}  {"Fee / contract":>16}  {"Fee as % of NO price"}')
print('  ' + '-' * 70)
for p in [0.03, 0.05, 0.06, 0.08, 0.10, 0.12, 0.15, 0.50]:
    fee = 0.07 * p * (1 - p)
    no_p = 1 - p
    print(f'  {p:>12.2f}  {no_p:>15.2f}  ${fee:>14.5f}  {fee/no_p*100:.3f}%')
print()
print('Key: fee is tiny for longshots (P×(1−P) small near 0). The relevant')
print('cost is the unobservable half-spread, captured by sensitivity grid below.')


Kalshi taker fee  =  0.07 × P × (1 − P)  per contract

   YES price P   NO price (1-P)    Fee / contract  Fee as % of NO price
  ----------------------------------------------------------------------
          0.03             0.97  $       0.00204  0.210%
          0.05             0.95  $       0.00333  0.350%
          0.06             0.94  $       0.00395  0.420%
          0.08             0.92  $       0.00515  0.560%
          0.10             0.90  $       0.00630  0.700%
          0.12             0.88  $       0.00739  0.840%
          0.15             0.85  $       0.00893  1.050%
          0.50             0.50  $       0.01750  3.500%

Key: fee is tiny for longshots (P×(1−P) small near 0). The relevant
cost is the unobservable half-spread, captured by sensitivity grid below.


In [3]:
# ── Universe: above-floor longshots ──────────────────────────────────────
print(f'Rows before filter : {len(cand):,}')
ab = cand[(cand['snapshot_price'] > FLOOR_MAX) & (cand['snapshot_price'] <= LS_MAX)].copy()
ab = ab.reset_index(drop=True)
print(f'Rows after  filter : {len(ab):,}  (0.02 < snapshot_price ≤ 0.15)')
print()

# PnL components  (one NO position per candidate, 1 contract, held to settlement)
ab['p']          = ab['snapshot_price']
ab['entry_cost'] = 1.0 - ab['p']                         # NO price = what you pay
ab['payoff']     = (1 - ab['won']).astype(float)          # $1 if candidate LOST
ab['gross_pnl']  = ab['p'] - ab['won'].astype(float)      # = payoff − entry_cost
ab['fee']        = FEE_RATE * ab['p'] * (1.0 - ab['p'])  # taker fee per contract

N       = len(ab)
N_ev    = ab['event'].nunique()
N_upset = int(ab['won'].sum())   # events where an above-floor longshot actually won

mean_p    = float(ab['p'].mean())
mean_won  = float(ab['won'].mean())
mean_cost = float(ab['entry_cost'].mean())
mean_gp   = float(ab['gross_pnl'].mean())   # = mean_p − mean_won
mean_fee  = float(ab['fee'].mean())
gross_roc = mean_gp / mean_cost

print(f'N candidates  : {N}')
print(f'N events      : {N_ev}')
print(f'N upset wins  : {N_upset}  (above-floor longshot was the actual winner)')
print()
print(f'Mean YES price (implied)   : {mean_p*100:.3f}¢')
print(f'Mean entry cost (NO price) : {mean_cost*100:.3f}¢')
print(f'Realized win rate          : {mean_won*100:.3f}%')
print()
print(f'Mean GROSS edge            : {mean_gp*100:.3f}¢  per contract')
print(f'Gross return on capital    : {gross_roc*100:.3f}%  (mean gross PnL / mean entry cost)')
print(f'Mean taker fee             : {mean_fee*100:.3f}¢  per contract')
breakeven_c = mean_gp - mean_fee
print(f'Breakeven half-spread c*   : {breakeven_c*100:.3f}¢  per contract  (gross edge − fee)')


Rows before filter : 1,579
Rows after  filter : 369  (0.02 < snapshot_price ≤ 0.15)

N candidates  : 369
N events      : 104
N upset wins  : 10  (above-floor longshot was the actual winner)

Mean YES price (implied)   : 6.643¢
Mean entry cost (NO price) : 93.357¢
Realized win rate          : 2.710%

Mean GROSS edge            : 3.933¢  per contract
Gross return on capital    : 4.212%  (mean gross PnL / mean entry cost)
Mean taker fee             : 0.426¢  per contract
Breakeven half-spread c*   : 3.507¢  per contract  (gross edge − fee)


In [4]:
# ── Event-clustered block bootstrap + cost sensitivity grid ───────────────
C_GRID = [0.000, 0.005, 0.010, 0.015, 0.020]

ev_groups = {ev: g.reset_index(drop=True) for ev, g in ab.groupby('event')}
evs = list(ev_groups.keys())

np.random.seed(RNG_SEED)
boot_gp  = np.empty(BOOT_REPS)
boot_fee = np.empty(BOOT_REPS)
boot_ec  = np.empty(BOOT_REPS)

for r in range(BOOT_REPS):
    samp = np.random.choice(evs, size=len(evs), replace=True)
    bdf  = pd.concat([ev_groups[e] for e in samp], ignore_index=True)
    boot_gp[r]  = float(bdf['gross_pnl'].mean())
    boot_fee[r] = float(bdf['fee'].mean())
    boot_ec[r]  = float(bdf['entry_cost'].mean())

print('=' * 88)
print('COST SENSITIVITY GRID — equal-weighted NO positions, 1 contract per candidate')
print('net_pnl = gross_pnl − taker_fee − c   (c = half-spread, per-contract entry cost)')
print('=' * 88)
hdr = (f'  {"c (half-spread)":>16}  {"Net edge (¢)":>13}  {"Net RoC (%)":>12}  '
       f'{"Clustered 95% CI":^26}  Sig?')
print(hdr)
print('  ' + '-' * 84)

grid_results = []
for c in C_GRID:
    pt_net = mean_gp - mean_fee - c
    pt_roc = pt_net / mean_cost
    boot_net = boot_gp - boot_fee - c
    ci_lo, ci_hi = np.percentile(boot_net, [2.5, 97.5])
    sig = 'YES' if ci_lo > 0 else 'no '
    print(f'  c = ${c:.3f}            {pt_net*100:>+10.3f}¢  {pt_roc*100:>11.3f}%  '
          f'[{ci_lo*100:+.3f}¢, {ci_hi*100:+.3f}¢]       {sig}')
    grid_results.append(dict(c=c, net_edge=pt_net, net_roc=pt_roc,
                              ci_lo=ci_lo, ci_hi=ci_hi, sig=(ci_lo > 0)))

print()
print(f'Breakeven c* = {breakeven_c*100:.3f}¢  (strategy earns zero net of fee at this spread cost)')


COST SENSITIVITY GRID — equal-weighted NO positions, 1 contract per candidate
net_pnl = gross_pnl − taker_fee − c   (c = half-spread, per-contract entry cost)
   c (half-spread)   Net edge (¢)   Net RoC (%)       Clustered 95% CI       Sig?
  ------------------------------------------------------------------------------------
  c = $0.000                +3.507¢        3.756%  [+1.940¢, +4.991¢]       YES
  c = $0.005                +3.007¢        3.220%  [+1.440¢, +4.491¢]       YES
  c = $0.010                +2.507¢        2.685%  [+0.940¢, +3.991¢]       YES
  c = $0.015                +2.007¢        2.149%  [+0.440¢, +3.491¢]       YES
  c = $0.020                +1.507¢        1.614%  [-0.060¢, +2.991¢]       no 

Breakeven c* = 3.507¢  (strategy earns zero net of fee at this spread cost)
